# Mining the PDB for Matched Cryo + Ambient-Temperature Pairs

**Project:** Beyond Cryo — predicting temperature-sensitive residues from sequence (Deep Learning in Genomics, final project).

**Goal:** Build a per-residue B-factor difference dataset by mining the RCSB PDB for proteins that have **both** a cryogenic (<150 K) and a near-physiological (>250 K) X-ray structure that share the same crystal form (same UniProt, same space group, unit cell within 5%, resolution < 2.5 Å).

**Inputs:** none — queries the public RCSB Search and Data APIs.

**Outputs:** a `data/` directory in the working directory containing:
- `data/manifest.csv` — one row per matched pair with metadata
- `data/sequences/{uniprot}.fasta` — canonical sequences
- `data/labels/{pair_id}.npz` — per-residue arrays: `b_cryo_z`, `b_ambient_z`, `delta_b`, `mask`, `seq_idx`
- `data/structures/{pdb_id}.cif` — cached mmCIF files
- `data/cache/*.json` — cached API responses so re-runs are fast

**AdoMetDC (UniProt P17707, 9P1H/9P7Q/9PBB) is excluded** — held out as the independent test case.

> Run cells top-to-bottom. The first run takes a few minutes; re-runs are fast (everything is cached).

## Glossary (for readers new to structural crystallography)

**Protein Data Bank (PDB / RCSB).** Public archive of experimentally determined macromolecular structures, accessed via two APIs: the **Search API** (find entries matching criteria) and the **Data API / GraphQL** (fetch metadata for known IDs). Each entry has a 4-character ID like `9P1H`.

**mmCIF.** The text file format for PDB entries (successor to the legacy PDB format). Parsed here with `gemmi`.

**UniProt accession.** Stable protein identifier (e.g., `P17707` for AdoMetDC). Two PDB entries that share an accession are structures of the same protein — that's how we match cryo and ambient pairs.

**Resolution (Å).** How fine the diffraction data is — *lower number = better*. We require ≤ 2.5 Å so per-residue B-factors are meaningful.

**Unit cell.** The repeating box of the crystal, defined by edge lengths `a, b, c` (Å) and angles `α, β, γ`. Two crystals "in the same crystal form" have nearly identical unit cells.

**Space group.** The symmetry of the crystal lattice (e.g., `P 1 21 1`, `P 21 21 21`) — one of 230 possibilities. Space group + unit cell together specify the crystal packing. We require both to match between cryo and ambient pairs so any B-factor difference reflects **temperature**, not different packing environments.

**Diffraction temperature (`diffrn.ambient_temp`).** The temperature (in Kelvin) at which the X-ray data were collected. The whole project hinges on this field: < 150 K is "cryo", > 250 K is "ambient" / near-physiological.

**Cryogenic crystallography (~100 K).** The default mode since the 1990s. Flash-cooling suppresses radiation damage but also **freezes out conformational motion** — flexible loops collapse into an artificial conformation, or disappear from electron density entirely, *looking* disordered when they're actually mobile at body temperature.

**Ambient / room-temperature crystallography (≥ 273 K).** Data collected near physiological temperature. Captures motions cryo cooling suppresses, but is much rarer in the PDB — which is why the ambient set is the rate-limiter for this dataset.

**B-factor (atomic displacement parameter / Debye–Waller factor).** Per-atom number quantifying how much an atom is "smeared out" in the electron density. It mixes true thermal motion, static conformational disorder, and refinement quirks. Higher = more mobile/uncertain. **Absolute B-factors are not comparable across crystals** (different overall scaling, mosaicity, and refinement choices), which is why we z-score within each structure before differencing cryo and ambient.

**Backbone atoms.** The four heavy atoms common to every standard amino acid: N, Cα, C, O. We average B-factor over these to get one number per residue.

**Modeled vs missing residue.** Crystallographers only build atoms into electron density they can interpret. A residue with no electron density → no atoms in the file → "missing" / "disordered" in the *structural* sense. This is distinct from **intrinsic disorder**: a residue can be missing in cryo but modeled in ambient, or missing in both.

**Author seq ID (`auth_seq_id`).** The residue number used by the depositing author — typically aligned to UniProt numbering. We use it to align residues between two structures of the same protein.

**Polymer entity.** A unique polymer sequence within a PDB entry. One entity can map to multiple physical chains (`auth_asym_ids`) — e.g., a homodimer has 2 chains but 1 polymer entity. AdoMetDC has 2 entities (α and β subunits) because it's autocatalytically cleaved from a single polypeptide.

## 1. Configuration

Tweak thresholds here. Defaults match the project plan.

In [1]:
from pathlib import Path

# --- Filtering thresholds ---
CRYO_TEMP_K        = 150.0   # diffrn.ambient_temp <  this  -> "cryo"
AMBIENT_TEMP_K     = 250.0   # diffrn.ambient_temp >  this  -> "ambient"
MAX_RESOLUTION_A   = 2.5     # Angstrom
CELL_TOL_FRAC      = 0.05    # 5% per axis (a, b, c)

# --- Held-out proteins (do not put in training set) ---
HELD_OUT_UNIPROTS  = {"P17707"}  # AdoMetDC
HELD_OUT_PDB_IDS   = {"9P1H", "9P7Q", "9PBB"}

# --- API endpoints ---
SEARCH_URL  = "https://search.rcsb.org/rcsbsearch/v2/query"
GRAPHQL_URL = "https://data.rcsb.org/graphql"
MMCIF_URL   = "https://files.rcsb.org/download/{pdb_id}.cif"

# --- Output layout (relative to the notebook's working directory) ---
DATA_DIR    = Path("data").resolve()
CACHE_DIR   = DATA_DIR / "cache"
STRUCT_DIR  = DATA_DIR / "structures"
LABELS_DIR  = DATA_DIR / "labels"
SEQ_DIR     = DATA_DIR / "sequences"

for d in (DATA_DIR, CACHE_DIR, STRUCT_DIR, LABELS_DIR, SEQ_DIR):
    d.mkdir(parents=True, exist_ok=True)

# --- Politeness / batch sizes ---
GRAPHQL_BATCH        = 100   # PDB IDs per GraphQL request
SEARCH_PAGE_SIZE     = 10000
UNIPROT_QUERY_CHUNK  = 100   # UniProt IDs per cryo-subset search query
REQUEST_TIMEOUT      = 60

print(f"Output directory: {DATA_DIR}")

Output directory: C:\Users\Jenitha Patel\OneDrive - The University of Chicago\S26\DL_genomics\final project\data


## 2. Imports & helpers

In [2]:
import json, time, sys, urllib.request
from typing import Iterable

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

try:
    import gemmi
except ImportError:
    print("Installing gemmi (used for mmCIF parsing)...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "gemmi"])
    import gemmi

print("gemmi version:", gemmi.__version__)


def post_json(url: str, payload: dict, timeout: int = REQUEST_TIMEOUT) -> dict:
    req = urllib.request.Request(
        url,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read())


def cached_json(path: Path):
    if path.exists():
        with path.open() as f:
            return json.load(f)
    return None


def write_json(path: Path, obj) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w") as f:
        json.dump(obj, f)


def chunks(seq, n):
    seq = list(seq)
    for i in range(0, len(seq), n):
        yield seq[i:i + n]

c:\Users\Jenitha Patel\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Installing gemmi (used for mmCIF parsing)...
gemmi version: 0.7.5


## 3. Query the RCSB Search API

Two queries: cryo (<150 K) and ambient (>250 K), both X-ray and resolution < 2.5 Å. The cryo set is huge (~140k); we don't enrich all of it. Instead we enrich the much smaller **ambient** set first, collect its UniProt accessions, then run a second cryo search constrained to those UniProts.

In [3]:
def search_pdb(temp_op: str, temp_value: float) -> list:
    # Return list of PDB IDs matching X-ray + temp + resolution filters.
    out: list = []
    start = 0
    while True:
        payload = {
            "query": {
                "type": "group",
                "logical_operator": "and",
                "nodes": [
                    {"type": "terminal", "service": "text",
                     "parameters": {"attribute": "exptl.method",
                                    "operator": "exact_match",
                                    "value": "X-RAY DIFFRACTION"}},
                    {"type": "terminal", "service": "text",
                     "parameters": {"attribute": "diffrn.ambient_temp",
                                    "operator": temp_op,
                                    "value": temp_value}},
                    {"type": "terminal", "service": "text",
                     "parameters": {"attribute": "rcsb_entry_info.resolution_combined",
                                    "operator": "less_or_equal",
                                    "value": MAX_RESOLUTION_A}},
                ],
            },
            "return_type": "entry",
            "request_options": {
                "paginate": {"start": start, "rows": SEARCH_PAGE_SIZE},
                "results_content_type": ["experimental"],
            },
        }
        data = post_json(SEARCH_URL, payload)
        hits = data.get("result_set", []) or []
        if not hits:
            break
        out.extend(h["identifier"] for h in hits)
        total = data.get("total_count", 0)
        start += len(hits)
        if start >= total:
            break
    return out


cache_cryo    = CACHE_DIR / "cryo_ids.json"
cache_ambient = CACHE_DIR / "ambient_ids.json"

cached = cached_json(cache_ambient)
if cached is not None:
    ambient_ids = cached
    print(f"Loaded {len(ambient_ids):,} ambient PDB IDs from cache")
else:
    print("Querying ambient (T > 250 K) ...")
    ambient_ids = search_pdb("greater", AMBIENT_TEMP_K)
    write_json(cache_ambient, ambient_ids)
    print(f"  -> {len(ambient_ids):,} ambient candidates")

cached = cached_json(cache_cryo)
if cached is not None:
    cryo_ids = cached
    print(f"Loaded {len(cryo_ids):,} cryo PDB IDs from cache (universe size, for reference)")
else:
    print("Querying cryo (T < 150 K) — this is large, may take ~30 s ...")
    cryo_ids = search_pdb("less", CRYO_TEMP_K)
    write_json(cache_cryo, cryo_ids)
    print(f"  -> {len(cryo_ids):,} cryo candidates (universe)")

Querying ambient (T > 250 K) ...
  -> 9,550 ambient candidates
Querying cryo (T < 150 K) — this is large, may take ~30 s ...
  -> 142,402 cryo candidates (universe)


## 4. Enrich ambient candidates with metadata

For each ambient PDB ID we batch-fetch UniProt accession(s), space group, unit cell, resolution, and ambient temperature via the GraphQL Data API. Cached on disk.

In [4]:
GRAPHQL_QUERY = """
query ($ids: [String!]!) {
  entries(entry_ids: $ids) {
    rcsb_id
    cell { length_a length_b length_c angle_alpha angle_beta angle_gamma }
    symmetry { space_group_name_H_M }
    rcsb_entry_info { resolution_combined }
    diffrn { ambient_temp }
    polymer_entities {
      rcsb_id
      entity_poly { pdbx_seq_one_letter_code_can }
      rcsb_polymer_entity_container_identifiers {
        auth_asym_ids
        reference_sequence_identifiers { database_accession database_name }
      }
    }
  }
}
""".strip()


def fetch_metadata(pdb_ids: list, cache_name: str) -> dict:
    # Batch GraphQL fetch. Returns {pdb_id: entry_dict}. Caches incrementally.
    cache_path = CACHE_DIR / cache_name
    cache: dict = cached_json(cache_path) or {}
    todo = [pid for pid in pdb_ids if pid not in cache]
    if todo:
        for batch in tqdm(list(chunks(todo, GRAPHQL_BATCH)), desc=f"GraphQL ({cache_name})"):
            data = post_json(GRAPHQL_URL, {"query": GRAPHQL_QUERY, "variables": {"ids": batch}})
            for entry in data.get("data", {}).get("entries", []) or []:
                if entry and entry.get("rcsb_id"):
                    cache[entry["rcsb_id"]] = entry
            write_json(cache_path, cache)  # checkpoint after each batch
            time.sleep(0.05)
    return {pid: cache[pid] for pid in pdb_ids if pid in cache}


ambient_meta = fetch_metadata(ambient_ids, "ambient_meta.json")
print(f"Got metadata for {len(ambient_meta):,} ambient entries")

GraphQL (ambient_meta.json): 100%|██████████| 96/96 [01:10<00:00,  1.36it/s]

Got metadata for 9,550 ambient entries


## 5. Index ambient entries by UniProt accession

Each PDB entry can map multiple polymer entities to multiple UniProts. We keep one record per (PDB, entity, UniProt) so we can match by protein.

In [5]:
def extract_entity_uniprots(entry: dict) -> list:
    rows = []
    for pe in entry.get("polymer_entities") or []:
        ids = pe.get("rcsb_polymer_entity_container_identifiers") or {}
        refs = ids.get("reference_sequence_identifiers") or []
        seq = (pe.get("entity_poly") or {}).get("pdbx_seq_one_letter_code_can") or ""
        chains = ids.get("auth_asym_ids") or []
        for ref in refs:
            if ref.get("database_name") == "UniProt":
                rows.append({
                    "pdb_id":  entry["rcsb_id"],
                    "entity_id": pe.get("rcsb_id"),
                    "uniprot": ref.get("database_accession"),
                    "chains":  chains,
                    "sequence": seq,
                })
    return rows


def cell_tuple(entry: dict):
    c = entry.get("cell") or {}
    return (c.get("length_a"), c.get("length_b"), c.get("length_c"),
            c.get("angle_alpha"), c.get("angle_beta"), c.get("angle_gamma"))


def space_group(entry: dict):
    return ((entry.get("symmetry") or {}).get("space_group_name_H_M") or "").strip()


def temp_value(entry: dict):
    diffs = entry.get("diffrn") or []
    temps = [d.get("ambient_temp") for d in diffs if d.get("ambient_temp") is not None]
    return float(temps[0]) if temps else None


def resolution(entry: dict):
    res = (entry.get("rcsb_entry_info") or {}).get("resolution_combined") or []
    return float(res[0]) if res else None


def filter_entries(meta: dict, low: float, high: float) -> list:
    """low < temp < high; also enforces resolution and held-out lists."""
    out = []
    for pid, entry in meta.items():
        if pid in HELD_OUT_PDB_IDS:
            continue
        t = temp_value(entry)
        if t is None or not (low < t < high):
            continue
        sg = space_group(entry)
        cell = cell_tuple(entry)
        if not sg or not all(cell):
            continue
        res = resolution(entry)
        if res is None or res > MAX_RESOLUTION_A:
            continue
        for r in extract_entity_uniprots(entry):
            if r["uniprot"] in HELD_OUT_UNIPROTS:
                continue
            r.update({"space_group": sg, "cell": cell, "resolution": res, "temp": t})
            out.append(r)
    return out


ambient_rows = filter_entries(ambient_meta, AMBIENT_TEMP_K, 1e6)
ambient_uniprots = sorted({r["uniprot"] for r in ambient_rows})
print(f"Ambient (post-filter): {len(ambient_rows):,} entity-UniProt records, "
      f"{len(ambient_uniprots):,} unique UniProts")

Ambient (post-filter): 11,487 entity-UniProt records, 2,958 unique UniProts


## 6. Find cryo entries for those UniProts

Re-run the search with an extra filter: the entry must reference a UniProt that already appears in the ambient set. We chunk the UniProt list because the search API has a per-query value-list limit.

In [6]:
def search_cryo_for_uniprots(uniprots: list) -> list:
    out: set = set()
    for chunk in tqdm(list(chunks(uniprots, UNIPROT_QUERY_CHUNK)), desc="Cryo search by UniProt"):
        start = 0
        while True:
            payload = {
                "query": {
                    "type": "group",
                    "logical_operator": "and",
                    "nodes": [
                        {"type": "terminal", "service": "text",
                         "parameters": {"attribute": "exptl.method",
                                        "operator": "exact_match",
                                        "value": "X-RAY DIFFRACTION"}},
                        {"type": "terminal", "service": "text",
                         "parameters": {"attribute": "diffrn.ambient_temp",
                                        "operator": "less",
                                        "value": CRYO_TEMP_K}},
                        {"type": "terminal", "service": "text",
                         "parameters": {"attribute": "rcsb_entry_info.resolution_combined",
                                        "operator": "less_or_equal",
                                        "value": MAX_RESOLUTION_A}},
                        {"type": "terminal", "service": "text",
                         "parameters": {
                            "attribute": "rcsb_polymer_entity_container_identifiers."
                                         "reference_sequence_identifiers.database_accession",
                            "operator": "in",
                            "value": chunk}},
                    ],
                },
                "return_type": "entry",
                "request_options": {
                    "paginate": {"start": start, "rows": SEARCH_PAGE_SIZE},
                    "results_content_type": ["experimental"],
                },
            }
            data = post_json(SEARCH_URL, payload)
            hits = data.get("result_set", []) or []
            if not hits:
                break
            out.update(h["identifier"] for h in hits)
            total = data.get("total_count", 0)
            start += len(hits)
            if start >= total:
                break
        time.sleep(0.05)
    return sorted(out)


cache_cryo_subset = CACHE_DIR / "cryo_subset_ids.json"
cached = cached_json(cache_cryo_subset)
if cached is not None:
    cryo_subset_ids = cached
else:
    cryo_subset_ids = search_cryo_for_uniprots(ambient_uniprots)
    write_json(cache_cryo_subset, cryo_subset_ids)
print(f"Cryo entries that share a UniProt with the ambient set: {len(cryo_subset_ids):,}")

cryo_meta = fetch_metadata(cryo_subset_ids, "cryo_meta.json")
cryo_rows = filter_entries(cryo_meta, -1e6, CRYO_TEMP_K)
print(f"Cryo (post-filter): {len(cryo_rows):,} entity-UniProt records")

Cryo search by UniProt: 100%|██████████| 30/30 [00:10<00:00,  2.90it/s]


Cryo entries that share a UniProt with the ambient set: 38,525


GraphQL (cryo_meta.json): 100%|██████████| 386/386 [11:37<00:00,  1.81s/it]


Cryo (post-filter): 49,707 entity-UniProt records


## 7. Match pairs by UniProt + space group + 5% cell

For each UniProt that appears in both sets, find the cryo/ambient pair that:
- shares the same Hermann–Mauguin space group symbol (whitespace-collapsed)
- has unit cell lengths a/b/c within 5% on each axis

If a UniProt has multiple valid pairs, keep the one with the **closest cell match**, breaking ties by best (lowest) average resolution.

In [7]:
from collections import defaultdict


def cell_matches(c1, c2, tol=CELL_TOL_FRAC) -> bool:
    a1, b1, cc1, *_ = c1
    a2, b2, cc2, *_ = c2
    for x, y in ((a1, a2), (b1, b2), (cc1, cc2)):
        if max(x, y) == 0:
            return False
        if abs(x - y) / max(x, y) > tol:
            return False
    return True


def cell_distance(c1, c2) -> float:
    return sum(abs(c1[i] - c2[i]) / max(c1[i], c2[i]) for i in range(3))


def sg_norm(sg: str) -> str:
    return " ".join(sg.split())


by_uniprot_amb = defaultdict(list)
by_uniprot_cryo = defaultdict(list)
for r in ambient_rows:
    by_uniprot_amb[r["uniprot"]].append(r)
for r in cryo_rows:
    by_uniprot_cryo[r["uniprot"]].append(r)

shared = sorted(set(by_uniprot_amb) & set(by_uniprot_cryo))
print(f"UniProts present in BOTH sets: {len(shared)}")

pairs = []
for up in shared:
    best = None
    for a in by_uniprot_amb[up]:
        for c in by_uniprot_cryo[up]:
            if sg_norm(a["space_group"]) != sg_norm(c["space_group"]):
                continue
            if not cell_matches(a["cell"], c["cell"]):
                continue
            score = cell_distance(a["cell"], c["cell"]) + (a["resolution"] + c["resolution"]) / 100.0
            if best is None or score < best[0]:
                best = (score, a, c)
    if best is not None:
        _, a, c = best
        pairs.append({
            "uniprot": up,
            "pdb_cryo": c["pdb_id"], "entity_cryo": c["entity_id"], "chains_cryo": c["chains"],
            "pdb_ambient": a["pdb_id"], "entity_ambient": a["entity_id"], "chains_ambient": a["chains"],
            "space_group": sg_norm(a["space_group"]),
            "cell_cryo": c["cell"], "cell_ambient": a["cell"],
            "resolution_cryo": c["resolution"], "resolution_ambient": a["resolution"],
            "temp_cryo": c["temp"], "temp_ambient": a["temp"],
            "sequence": a["sequence"] or c["sequence"],
        })

print(f"Matched pairs after filters: {len(pairs)}")
pd.DataFrame([{k: v for k, v in p.items() if k != "sequence"} for p in pairs]).head(10)

UniProts present in BOTH sets: 1899
Matched pairs after filters: 1162


,uniprot,pdb_cryo,entity_cryo,chains_cryo,pdb_ambient,entity_ambient,chains_ambient,space_group,cell_cryo,cell_ambient,resolution_cryo,resolution_ambient,temp_cryo,temp_ambient
0,A0A075B6C4,7N5P,7N5P_4,[D],7N4K,7N4K_4,[D],P 1 21 1,"(54.334, 72.056, 107.671, 90.0, 101.28, 90.0)","(54.14, 72.53, 107.72, 90.0, 101.36, 90.0)",2.09,1.85,100.0,277.00
1,A0A086IRG1,9RQ3,9RQ3_1,"[A, B]",9S5B,9S5B_1,"[A, B]",P 21 21 2,"(68.278, 89.802, 44.974, 90.0, 90.0, 90.0)","(69.28, 90.83, 45.26, 90.0, 90.0, 90.0)",1.22,1.50,100.0,296.00
2,A0A0A1EQY0,8HTY,8HTY_1,"[A, B, C, D]",8IQ7,8IQ7_1,"[A, B, C, D]",P 1,"(54.132, 68.959, 109.792, 78.14, 89.48, 81.11)","(54.668, 69.509, 110.8, 77.86, 89.05, 81.2)",1.40,2.10,100.0,298.00
3,A0A0C3QM78,9H06,9H06_1,[A],9H07,9H07_1,[A],P 43 21 2,"(59.971, 59.971, 231.864, 90.0, 90.0, 90.0)","(59.959, 59.959, 231.749, 90.0, 90.0, 90.0)",1.86,2.41,100.0,293.00
4,A0A0E1CQ35,9G1M,9G1M_1,"[A, B]",9G1E,9G1E_1,"[A, B]",P 21 21 2,"(68.15, 89.792, 44.998, 90.0, 90.0, 90.0)","(69.35, 90.88, 45.3, 90.0, 90.0, 90.0)",1.18,1.40,100.0,296.15
5,A0A0G2LB96,7N5P,7N5P_5,[E],7N4K,7N4K_5,[E],P 1 21 1,"(54.334, 72.056, 107.671, 90.0, 101.28, 90.0)","(54.14, 72.53, 107.72, 90.0, 101.36, 90.0)",2.09,1.85,100.0,277.00
6,A0A0H2XJL0,9C0L,9C0L_1,[A],9C0M,9C0M_1,[A],P 43 21 2,"(60.898, 60.898, 164.513, 90.0, 90.0, 90.0)","(62.285, 62.285, 164.094, 90.0, 90.0, 90.0)",1.79,2.50,100.0,295.00
7,A0A0H2Z117,8ZVB,8ZVB_1,[A],8ZVD,8ZVD_1,[A],P 1 21 1,"(49.381, 64.736, 64.473, 90.0, 91.123, 90.0)","(49.848, 64.288, 64.401, 90.0, 90.578, 90.0)",2.04,1.72,100.0,289.00
8,A0A0H3AFW5,4I46,4I46_1,"[A, B, C, D, E, F]",5JZO,5JZO_1,"[A, B, C, D, E, F]",P 1 21 1,"(73.287, 80.03, 133.037, 90.0, 95.36, 90.0)","(73.375, 80.069, 133.133, 90.0, 95.35, 90.0)",2.50,2.50,100.0,293.00
9,A0A0H3AJ04,6IH1,6IH1_1,"[A, B, C, D]",6K6T,6K6T_1,"[A, B, C, D]",P 1 21 1,"(73.5, 42.63, 156.34, 90.0, 94.42, 90.0)","(73.45, 42.54, 155.04, 90.0, 94.65, 90.0)",1.95,2.20,100.0,273.00


## 8. Download mmCIF files

Cache full mmCIFs in `data/structures/`. Re-runs skip already-downloaded files.

In [8]:
def download_cif(pdb_id: str) -> Path:
    out = STRUCT_DIR / f"{pdb_id}.cif"
    if out.exists() and out.stat().st_size > 0:
        return out
    url = MMCIF_URL.format(pdb_id=pdb_id)
    with urllib.request.urlopen(url, timeout=REQUEST_TIMEOUT) as r:
        out.write_bytes(r.read())
    return out


to_dl = sorted({p["pdb_cryo"] for p in pairs} | {p["pdb_ambient"] for p in pairs})
for pid in tqdm(to_dl, desc="Downloading mmCIFs"):
    try:
        download_cif(pid)
    except Exception as e:
        print(f"  ! {pid}: {e}")
print(f"Cached {len(to_dl)} mmCIF files in {STRUCT_DIR}")

Cached 2068 mmCIF files in C:\Users\Jenitha Patel\OneDrive - The University of Chicago\S26\DL_genomics\final project\data\structures


## 9. Per-residue B-factor extraction & label computation

For each pair we pick the chain corresponding to the UniProt-mapped polymer entity, then collect per-residue mean B-factors over backbone heavy atoms (N, Cα, C, O). B-factors are **z-scored within each structure** before differencing — absolute B-factors aren't comparable across crystals (different scaling, mosaicity, etc.).

Author seq IDs anchor residues. A residue modeled in one structure but missing in the other becomes a NaN entry on the missing side; the mask records this.

In [9]:
BACKBONE = {"N", "CA", "C", "O"}
THREE_TO_ONE = {
    "ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C",
    "GLN": "Q", "GLU": "E", "GLY": "G", "HIS": "H", "ILE": "I",
    "LEU": "L", "LYS": "K", "MET": "M", "PHE": "F", "PRO": "P",
    "SER": "S", "THR": "T", "TRP": "W", "TYR": "Y", "VAL": "V",
    "MSE": "M", "SEC": "U", "PYL": "O",
}


def parse_chain_bfactors(cif_path: Path, chain_ids: list):
    # Return (chain_id, {auth_seq_id: (one_letter, mean_backbone_B)}) for the FIRST chain
    # in chain_ids that exists in the structure.
    st = gemmi.read_structure(str(cif_path))
    st.setup_entities()
    model = st[0]
    for cid in chain_ids:
        chain = next((ch for ch in model if ch.name == cid), None)
        if chain is None:
            continue
        out = {}
        for res in chain:
            name = res.name.strip().upper()
            if name not in THREE_TO_ONE:
                continue
            bs = [a.b_iso for a in res if a.name in BACKBONE and a.element.name != "H"]
            if not bs:
                continue
            try:
                seq_id = int(res.seqid.num)
            except Exception:
                continue
            out[seq_id] = (THREE_TO_ONE[name], float(np.mean(bs)))
        if out:
            return cid, out
    return None, {}


def build_pair_labels(pair: dict):
    cif_a = STRUCT_DIR / f"{pair['pdb_ambient']}.cif"
    cif_c = STRUCT_DIR / f"{pair['pdb_cryo']}.cif"
    chain_a, res_a = parse_chain_bfactors(cif_a, pair["chains_ambient"])
    chain_c, res_c = parse_chain_bfactors(cif_c, pair["chains_cryo"])
    if not res_a or not res_c:
        return None

    def zscore(d):
        vals = np.array([v for _, v in d.values()], dtype=float)
        mu = float(vals.mean())
        sd = float(vals.std(ddof=0)) or 1.0
        return {k: (aa, (v - mu) / sd) for k, (aa, v) in d.items()}, mu, sd

    res_a_z, mu_a, sd_a = zscore(res_a)
    res_c_z, mu_c, sd_c = zscore(res_c)

    all_ids = sorted(set(res_a) | set(res_c))
    seq = []
    b_amb = np.full(len(all_ids), np.nan)
    b_cry = np.full(len(all_ids), np.nan)
    for i, sid in enumerate(all_ids):
        aa = (res_a.get(sid) or res_c.get(sid))[0]
        seq.append(aa)
        if sid in res_a_z:
            b_amb[i] = res_a_z[sid][1]
        if sid in res_c_z:
            b_cry[i] = res_c_z[sid][1]
    delta = b_amb - b_cry
    mask = ~np.isnan(delta)

    aa_disagreements = sum(
        1 for sid in (set(res_a) & set(res_c)) if res_a[sid][0] != res_c[sid][0]
    )

    return {
        "seq_idx": np.array(all_ids, dtype=int),
        "sequence": "".join(seq),
        "b_ambient_z": b_amb,
        "b_cryo_z": b_cry,
        "delta_b": delta,
        "mask": mask,
        "chain_ambient": chain_a, "chain_cryo": chain_c,
        "n_residues": len(all_ids),
        "n_modeled_both": int(mask.sum()),
        "n_only_ambient": int((~np.isnan(b_amb) & np.isnan(b_cry)).sum()),
        "n_only_cryo":    int((np.isnan(b_amb) & ~np.isnan(b_cry)).sum()),
        "aa_disagreements": aa_disagreements,
        "raw_mean_B_ambient": mu_a, "raw_std_B_ambient": sd_a,
        "raw_mean_B_cryo":    mu_c, "raw_std_B_cryo":    sd_c,
    }


pair_labels = []
for p in tqdm(pairs, desc="Extracting B-factors"):
    try:
        lab = build_pair_labels(p)
    except Exception as e:
        print(f"  ! {p['pdb_cryo']}/{p['pdb_ambient']}: {e}")
        continue
    if lab is None:
        continue
    pair_labels.append((p, lab))
print(f"Pairs with successful label extraction: {len(pair_labels)} / {len(pairs)}")

Extracting B-factors: 100%|██████████| 1162/1162 [05:35<00:00,  3.46it/s]

Pairs with successful label extraction: 1162 / 1162


## 10. Save labels, sequences, and manifest

In [10]:
def fasta_write(path: Path, header: str, sequence: str, width: int = 60) -> None:
    with path.open("w") as f:
        f.write(f">{header}\n")
        for i in range(0, len(sequence), width):
            f.write(sequence[i:i + width] + "\n")


manifest_rows = []
for p, lab in pair_labels:
    pair_id = f"{p['uniprot']}_{p['pdb_cryo']}_{p['pdb_ambient']}"
    np.savez(LABELS_DIR / f"{pair_id}.npz",
             seq_idx=lab["seq_idx"],
             b_cryo_z=lab["b_cryo_z"],
             b_ambient_z=lab["b_ambient_z"],
             delta_b=lab["delta_b"],
             mask=lab["mask"],
             sequence=np.array(lab["sequence"]))

    fasta_path = SEQ_DIR / f"{p['uniprot']}.fasta"
    if not fasta_path.exists() and p.get("sequence"):
        fasta_write(fasta_path, p["uniprot"], p["sequence"])

    a_cell = p["cell_ambient"]; c_cell = p["cell_cryo"]
    manifest_rows.append({
        "pair_id": pair_id,
        "uniprot": p["uniprot"],
        "pdb_cryo": p["pdb_cryo"], "chain_cryo": lab["chain_cryo"],
        "pdb_ambient": p["pdb_ambient"], "chain_ambient": lab["chain_ambient"],
        "space_group": p["space_group"],
        "cell_a_cryo": c_cell[0], "cell_b_cryo": c_cell[1], "cell_c_cryo": c_cell[2],
        "cell_a_ambient": a_cell[0], "cell_b_ambient": a_cell[1], "cell_c_ambient": a_cell[2],
        "max_cell_dev_pct": 100 * max(abs(c_cell[i] - a_cell[i]) / max(c_cell[i], a_cell[i]) for i in range(3)),
        "resolution_cryo": p["resolution_cryo"], "resolution_ambient": p["resolution_ambient"],
        "temp_cryo": p["temp_cryo"], "temp_ambient": p["temp_ambient"],
        "n_residues": lab["n_residues"],
        "n_modeled_both": lab["n_modeled_both"],
        "n_only_ambient": lab["n_only_ambient"],
        "n_only_cryo": lab["n_only_cryo"],
        "aa_disagreements": lab["aa_disagreements"],
        "labels_path": str((LABELS_DIR / f"{pair_id}.npz").relative_to(DATA_DIR)),
        "sequence_path": str(fasta_path.relative_to(DATA_DIR)),
    })

manifest = pd.DataFrame(manifest_rows).sort_values(["uniprot", "pair_id"]).reset_index(drop=True)
manifest_path = DATA_DIR / "manifest.csv"
manifest.to_csv(manifest_path, index=False)
print(f"Wrote {len(manifest)} rows to {manifest_path}")
manifest.head(10)

Wrote 1162 rows to C:\Users\Jenitha Patel\OneDrive - The University of Chicago\S26\DL_genomics\final project\data\manifest.csv


,pair_id,uniprot,pdb_cryo,chain_cryo,pdb_ambient,chain_ambient,space_group,cell_a_cryo,cell_b_cryo,cell_c_cryo,...,resolution_ambient,temp_cryo,temp_ambient,n_residues,n_modeled_both,n_only_ambient,n_only_cryo,aa_disagreements,labels_path,sequence_path
0,A0A075B6C4_7N5P_7N4K,A0A075B6C4,7N5P,D,7N4K,D,P 1 21 1,54.334,72.056,107.671,...,1.85,100.0,277.00,196,195,1,0,0,labels\A0A075B6C4_7N5P_7N4K.npz,sequences\A0A075B6C4.fasta
1,A0A086IRG1_9RQ3_9S5B,A0A086IRG1,9RQ3,A,9S5B,A,P 21 21 2,68.278,89.802,44.974,...,1.50,100.0,296.00,138,137,1,0,0,labels\A0A086IRG1_9RQ3_9S5B.npz,sequences\A0A086IRG1.fasta
2,A0A0A1EQY0_8HTY_8IQ7,A0A0A1EQY0,8HTY,A,8IQ7,A,P 1,54.132,68.959,109.792,...,2.10,100.0,298.00,356,356,0,0,0,labels\A0A0A1EQY0_8HTY_8IQ7.npz,sequences\A0A0A1EQY0.fasta
3,A0A0C3QM78_9H06_9H07,A0A0C3QM78,9H06,A,9H07,A,P 43 21 2,59.971,59.971,231.864,...,2.41,100.0,293.00,235,226,0,9,0,labels\A0A0C3QM78_9H06_9H07.npz,sequences\A0A0C3QM78.fasta
4,A0A0E1CQ35_9G1M_9G1E,A0A0E1CQ35,9G1M,A,9G1E,A,P 21 21 2,68.150,89.792,44.998,...,1.40,100.0,296.15,137,137,0,0,0,labels\A0A0E1CQ35_9G1M_9G1E.npz,sequences\A0A0E1CQ35.fasta
5,A0A0G2LB96_7N5P_7N4K,A0A0G2LB96,7N5P,E,7N4K,E,P 1 21 1,54.334,72.056,107.671,...,1.85,100.0,277.00,240,235,5,0,0,labels\A0A0G2LB96_7N5P_7N4K.npz,sequences\A0A0G2LB96.fasta
6,A0A0H2XJL0_9C0L_9C0M,A0A0H2XJL0,9C0L,A,9C0M,A,P 43 21 2,60.898,60.898,164.513,...,2.50,100.0,295.00,243,242,0,1,0,labels\A0A0H2XJL0_9C0L_9C0M.npz,sequences\A0A0H2XJL0.fasta
7,A0A0H2Z117_8ZVB_8ZVD,A0A0H2Z117,8ZVB,A,8ZVD,A,P 1 21 1,49.381,64.736,64.473,...,1.72,100.0,289.00,412,393,19,0,0,labels\A0A0H2Z117_8ZVB_8ZVD.npz,sequences\A0A0H2Z117.fasta
8,A0A0H3AFW5_4I46_5JZO,A0A0H3AFW5,4I46,A,5JZO,A,P 1 21 1,73.287,80.030,133.037,...,2.50,100.0,293.00,281,281,0,0,0,labels\A0A0H3AFW5_4I46_5JZO.npz,sequences\A0A0H3AFW5.fasta
9,A0A0H3AJ04_6IH1_6K6T,A0A0H3AJ04,6IH1,A,6K6T,A,P 1 21 1,73.500,42.630,156.340,...,2.20,100.0,273.00,237,237,0,0,0,labels\A0A0H3AJ04_6IH1_6K6T.npz,sequences\A0A0H3AJ04.fasta


## 11. Summary statistics

Quick look at the dataset to spot anomalies before training.

In [11]:
if len(manifest) == 0:
    print("No pairs survived filtering. Loosen thresholds and re-run.")
else:
    print(f"Pairs:                 {len(manifest)}")
    print(f"Unique UniProts:       {manifest['uniprot'].nunique()}")
    print(f"Median residues/pair:  {int(manifest['n_residues'].median())}")
    print(f"Median modeled/pair:   {int(manifest['n_modeled_both'].median())}")
    print(f"Median |Δ resolution|: "
          f"{(manifest['resolution_cryo'] - manifest['resolution_ambient']).abs().median():.2f} Å")
    print(f"Median Δ temperature:  "
          f"{(manifest['temp_ambient'] - manifest['temp_cryo']).median():.0f} K")
    print(f"Median max cell drift: {manifest['max_cell_dev_pct'].median():.2f} %")

    all_dB = []
    for _, row in manifest.iterrows():
        d = np.load(LABELS_DIR / f"{row['pair_id']}.npz")
        v = d["delta_b"]
        all_dB.append(v[~np.isnan(v)])
    all_dB = np.concatenate(all_dB) if all_dB else np.array([])
    if all_dB.size:
        print(f"\nΔ z-scored B-factor over all residues: "
              f"n={all_dB.size:,}, mean={all_dB.mean():+.3f}, std={all_dB.std():.3f}, "
              f"|q95|={np.quantile(np.abs(all_dB), 0.95):.3f}")

Pairs:                 1162
Unique UniProts:       1162
Median residues/pair:  244
Median modeled/pair:   237
Median |Δ resolution|: 0.29 Å
Median Δ temperature:  193 K
Median max cell drift: 0.96 %

Δ z-scored B-factor over all residues: n=295,821, mean=-0.002, std=0.553, |q95|=1.111


## Next steps

- `data/manifest.csv` is the index of usable pairs; per-residue labels are in `data/labels/`.
- When building train/val, **split by UniProt** (not by residue) to prevent leakage.
- AdoMetDC (P17707, 9P1H/9P7Q/9PBB) was excluded — keep it out of training and use it as the held-out independent test case.
- If pair count is too low, the easiest knobs to loosen (in this order): `MAX_RESOLUTION_A` (e.g., 2.8), `CELL_TOL_FRAC` (e.g., 0.10), then `AMBIENT_TEMP_K` down to 240 K.
- Want a missing-residue **binary** label too? It's already in each `.npz` as the mask of `b_ambient_z` vs `b_cryo_z`: residues where one is NaN and the other isn't are exactly the DL2-style "ordered at one temp, missing at the other" cases.